In [ ]:
import argparse
import sys
from pathlib import Path

import torch


from connectomics.config import load_config, DEFAULT_CONFIG
from connectomics.model import load_model, verify_tokenization, verify_answer_token, run_with_cache, sanity_check
from connectomics.attribution import (
    final_logit_margin, component_attribution, head_attribution,
    print_top_components, print_top_heads,
)
from connectomics.visualization import (
    plot_tokenization_panel, plot_component_attribution,
    plot_head_attribution, plot_top_head_attention, plot_attention_pattern
)

In [ ]:
from connectomics.model import gpu_memory, free_memory

# check state at any time
gpu_memory()

# after an experiment, pass whatever tensors/caches you're done with
free_memory(logits, cache)

# or just clear the CUDA allocator without deleting anything specific
free_memory()


In [ ]:
cfg = load_config()

In [ ]:
model = load_model(cfg.model)

In [ ]:
outdir = f"{cfg.output.plot_root}/01_baseline"
Path(outdir).mkdir(parents=True, exist_ok=True)

In [ ]:
cfg.tokens

In [ ]:
sanity_check(model, "1 + 1 = ")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print("\n--- Logit lens ---")

bos = cfg.model.prepend_bos
tokens, str_tokens = verify_tokenization(model, cfg.prompts.clean, prepend_bos=bos)
answer_id = verify_answer_token(model, cfg.tokens.answer)

logits, cache = model.run_with_cache(tokens)

In [ ]:
print("\n--- Forward pass ---")
logits, cache = run_with_cache(model, tokens, strategy=cfg.model.cache_strategy)
distractors = [cfg.tokens.distractor] if cfg.tokens.distractor else None
margin = final_logit_margin(model, logits, cfg.tokens.answer, distractors)
print("Final logit margin:")
for k, v in margin.items():
    print(f"  {k}: {v}")

print("\n--- Component attribution ---")
attrs, labels = component_attribution(model, cache, cfg.tokens.answer, pos=-1)
print_top_components(attrs, labels, repr(cfg.tokens.answer))
plot_component_attribution(attrs, labels, repr(cfg.tokens.answer),
                           outdir=outdir, filename="component_attribution.png")

print("\n--- Per-head attribution ---")
head_attrs = head_attribution(model, cache, cfg.tokens.answer, pos=-1)
print_top_heads(head_attrs, repr(cfg.tokens.answer))
plot_head_attribution(head_attrs, repr(cfg.tokens.answer),
                      outdir=outdir, filename="head_attribution.png")
plot_top_head_attention(model, cache, head_attrs, str_tokens, k=4,
                        outdir=outdir, filename_prefix="top_head_attn")

print(f"\nAll plots saved to {outdir}/")

In [ ]:
prompts = {"clean": cfg.prompts.clean, "corrupted": cfg.prompts.corrupted}
if cfg.prompts.control:
    prompts["control"] = cfg.prompts.control

results = {}
for label, prompt in prompts.items():
    print(f"\n{'='*40}\n  {label.upper()}: {repr(prompt)}\n{'='*40}")
    tokens, str_tokens = verify_tokenization(model, prompt, prepend_bos=bos)
    logits, cache = run_with_cache(model, tokens, strategy=cfg.model.cache_strategy)

    suffix_start = None
    if label != "clean":
        suffix_start = model.to_tokens(cfg.prompts.clean, prepend_bos=bos).shape[1]

    plot_tokenization_panel(str_tokens, tokens, suffix_start=suffix_start,
                            title=f"{label}: {repr(prompt)}",
                            outdir=outdir, filename=f"tokenization_{label}.png")

    distractors = [cfg.tokens.distractor] if cfg.tokens.distractor else None
    margin = final_logit_margin(model, logits, cfg.tokens.answer, distractors)
    print(f"  P({repr(cfg.tokens.answer)})={margin['prob']:.4f}  rank={margin['rank']}")

    results[label] = {"tokens": tokens, "str_tokens": str_tokens,
                      "logits": logits, "cache": cache}

# Head attribution
print("\n--- Head attribution comparison ---")
head_results = {}
for label, r in results.items():
    ha = head_attribution(model, r["cache"], cfg.tokens.answer, pos=-1)
    head_results[label] = ha
    plot_head_attribution(ha, f"{repr(cfg.tokens.answer)} ({label})",
                          outdir=outdir, filename=f"head_attribution_{label}.png")

diff = head_results["corrupted"] - head_results["clean"]
plot_head_attribution(diff, f"{repr(cfg.tokens.answer)} (corrupted - clean)",
                      outdir=outdir, filename="head_attribution_diff.png")
print_top_heads(diff, f"{repr(cfg.tokens.answer)} change (corrupted - clean)")

# Top changed heads attention patterns
n_heads = diff.shape[1]
flat = diff.flatten()
top_idx = flat.abs().argsort(descending=True)[:4]
for rank, idx in enumerate(top_idx):
    li, hi = idx.item() // n_heads, idx.item() % n_heads
    for label, r in results.items():
        plot_attention_pattern(r["cache"], li, hi, r["str_tokens"],
                              title=f"L{li} H{hi} ({label})",
                              outdir=outdir,
                              filename=f"changed_head_{rank}_L{li}_H{hi}_{label}.png")

print(f"\nAll plots saved to {outdir}/")
